# Module 6 — SCD Type 1/2, Schema Evolution
Exam domain: **Data Modeling**

Runs standalone in Google Colab — no Databricks account needed.

In [ ]:
!pip install -q pyspark==3.5.1 delta-spark==3.2.0

In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

builder = (SparkSession.builder
    .appName("Module6-SCD")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()

## SCD Type 1 — overwrite, no history
The old value is simply replaced. Good when you don't need to know what the
value used to be.

In [ ]:
dim_customer = spark.createDataFrame(
    [(1, "Alice", "Madrid"), (2, "Bob", "Lisbon")], ["customer_id", "name", "city"])
dim_customer.write.format("delta").mode("overwrite").save("/content/lake/dim_customer_scd1")

scd1_target = DeltaTable.forPath(spark, "/content/lake/dim_customer_scd1")
changes = spark.createDataFrame([(1, "Alice", "Barcelona")], ["customer_id", "name", "city"])  # Alice moved

(scd1_target.alias("t")
    .merge(changes.alias("s"), "t.customer_id = s.customer_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

spark.read.format("delta").load("/content/lake/dim_customer_scd1").show()  # Madrid is gone

## SCD Type 2 — full history with effective/end dates
Every change inserts a new row and closes out the old one; `is_current` marks
the active version.

In [ ]:
from pyspark.sql.types import DateType

dim_customer_scd2 = spark.createDataFrame(
    [(1, "Alice", "Madrid", "2024-01-01", None, True)],
    ["customer_id", "name", "city", "effective_date", "end_date", "is_current"]
).withColumn("effective_date", F.to_date("effective_date")) \
 .withColumn("end_date", F.col("end_date").cast(DateType()))

dim_customer_scd2.write.format("delta").mode("overwrite").save("/content/lake/dim_customer_scd2")

In [ ]:
def scd2_merge(table_path, changes_df, business_key, change_date):
    target = DeltaTable.forPath(spark, table_path)

    # Stage 1: close out rows whose tracked attributes changed
    staged = (changes_df.alias("s")
        .join(target.toDF().alias("t").filter("is_current = true"),
              on=business_key, how="inner")
        .filter("s.city <> t.city")  # the attribute(s) we're tracking history for
        .select("s.*"))

    (target.alias("t")
        .merge(staged.alias("s"), f"t.{business_key} = s.{business_key} AND t.is_current = true")
        .whenMatchedUpdate(set={"end_date": F.lit(change_date), "is_current": F.lit(False)})
        .execute())

    # Stage 2: insert the new current row
    new_rows = staged.withColumn("effective_date", F.lit(change_date)) \
                      .withColumn("end_date", F.lit(None).cast(DateType())) \
                      .withColumn("is_current", F.lit(True))
    new_rows.write.format("delta").mode("append").save(table_path)

changes = spark.createDataFrame([(1, "Alice", "Barcelona")], ["customer_id", "name", "city"])
scd2_merge("/content/lake/dim_customer_scd2", changes, "customer_id", "2024-03-01")

spark.read.format("delta").load("/content/lake/dim_customer_scd2").orderBy("effective_date").show()

## Schema evolution with `mergeSchema`
Delta refuses writes with a mismatched schema by default (safety). `mergeSchema`
opts in to appending new columns.

In [ ]:
base = spark.createDataFrame([(1, "Alice")], ["id", "name"])
base.write.format("delta").mode("overwrite").save("/content/lake/schema_evo_demo")

evolved = spark.createDataFrame([(2, "Bob", "gold")], ["id", "name", "tier"])
try:
    evolved.write.format("delta").mode("append").save("/content/lake/schema_evo_demo")
except Exception as e:
    print("Failed without mergeSchema:", str(e)[:200])

evolved.write.format("delta").mode("append").option("mergeSchema", "true").save("/content/lake/schema_evo_demo")
spark.read.format("delta").load("/content/lake/schema_evo_demo").show()  # id=1 has tier = null

## Interview-relevant contrast
- **SCD1**: cheap, no audit trail, breaks any report that relied on the old value.
- **SCD2**: full audit trail, more storage, every query on the dimension must
  filter `is_current = true` (or a date range) or it will double-count facts.